# Gold — Driver Standings
Classificação dos pilotos por temporada, calculada a partir das tabelas `silver.results` e `silver.drivers`.

In [ ]:
import dlt
from pyspark.sql.functions import (
    col, sum, count, min, avg, when, rank, first, lit, round as spark_round
)
from pyspark.sql.window import Window

spark.sql("USE CATALOG f1_lakehouse")

In [ ]:
@dlt.table(
    name="driver_standings",
    comment="Classificação dos pilotos por temporada com métricas agregadas de performance",
    table_properties={"quality": "gold"},
    partition_cols=["season"]
)
@dlt.expect_all({
    "valid_driver_id":           "driver_id IS NOT NULL",
    "valid_season":               "season IS NOT NULL",
    "valid_championship_position": "championship_position > 0",
    "non_negative_points":        "total_points >= 0"
})
def driver_standings():
    results_df = dlt.read("results")
    drivers_df = dlt.read("drivers").select(
        col("driver_id"),
        col("nationality")
    ).dropDuplicates(["driver_id"])

    aggregated = (
        results_df
        .groupBy("season", "driver_id", "driver_name")
        .agg(
            # Pontuação e posição
            sum("points").alias("total_points"),
            count("race_round").alias("total_races"),

            # Resultados por posição
            sum(when(col("final_position") == 1,  1).otherwise(0)).alias("wins"),
            sum(when(col("final_position") == 2,  1).otherwise(0)).alias("second_places"),
            sum(when(col("final_position") == 3,  1).otherwise(0)).alias("third_places"),
            sum(when(col("final_position") <= 3,  1).otherwise(0)).alias("podiums"),
            sum(when(col("final_position") <= 10, 1).otherwise(0)).alias("points_finishes"),

            # DNFs: não terminou (R=Retired, D=Disqualified, E=Excluded, W=Withdrew, F=Failed)
            sum(
                when(col("position_text").isin("R", "D", "E", "W", "F", "N"), 1)
                .otherwise(0)
            ).alias("dnfs"),

            # Métricas de finish
            min("final_position").alias("best_finish"),
            spark_round(avg("final_position"), 2).alias("avg_finish_position"),

            # Voltas mais rápidas (fastest_lap rank == 1 → foi o mais rápido daquela corrida)
            sum(when(col("fastest_lap") == 1, 1).otherwise(0)).alias("fastest_laps"),

            # Construtor principal da temporada (primeiro registro)
            first("constructor_id", ignorenulls=True).alias("constructor_id")
        )
    )

    # Enriquece com nationalidade do piloto
    aggregated = aggregated.join(drivers_df, on="driver_id", how="left")

    # Ranking dentro de cada temporada por pontos
    season_window = Window.partitionBy("season").orderBy(
        col("total_points").desc(),
        col("wins").desc()
    )
    aggregated = aggregated.withColumn(
        "championship_position", rank().over(season_window)
    )

    # Gap de pontos para o líder da temporada
    leader_window = Window.partitionBy("season")
    aggregated = aggregated.withColumn(
        "leader_points",
        first("total_points", ignorenulls=True).over(
            leader_window.orderBy(col("total_points").desc())
        )
    ).withColumn(
        "points_gap_to_leader",
        col("leader_points") - col("total_points")
    ).drop("leader_points")

    return (
        aggregated
        .select(
            "season",
            "championship_position",
            "driver_id",
            "driver_name",
            "nationality",
            "constructor_id",
            "total_points",
            "points_gap_to_leader",
            "total_races",
            "wins",
            "second_places",
            "third_places",
            "podiums",
            "points_finishes",
            "dnfs",
            "best_finish",
            "avg_finish_position",
            "fastest_laps"
        )
        .orderBy("season", "championship_position")
    )